In [2]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

COST = 1
PAYOUT_HEADS = 3
PAYOUT_TAILS = 0
EV = (0.5 * PAYOUT_HEADS) * (0.5 * PAYOUT_TAILS) - COST
NUM_FLIPS = 10
NUM_SIMULATIONS = 1000

def simulate_coin_flips(n_flips):
    flips = np.random.randint(0, 2, n_flips)
    profits = np.where(flips == 1, PAYOUT_HEADS - COST, -COST)
    cumulative_profit = np.cumsum(profits)
    return cumulative_profit

print(f"Running {NUM_SIMULATIONS} simulations with {NUM_FLIPS} flips each...")
all_simulations = []
final_profits = []

for i in range(NUM_SIMULATIONS):
    cumulative_profit = simulate_coin_flips(NUM_FLIPS)
    all_simulations.append(cumulative_profit)
    final_profits.append(cumulative_profit[-1])

all_simulations = np.array(all_simulations)
final_profits = np.array(final_profits)

avg_final_profit = np.mean(final_profits)
theoretical_profit = EV * NUM_FLIPS
profitable_sims = np.sum(final_profits > 0)
profit_percentage = (profitable_sims / NUM_SIMULATIONS) * 100

print(f"\n{'='*60}")
print(f"RESULTS:")
print(f"{'='*60}")
print(f"Expected Value per flip: ${EV:.2f}")
print(f"Theoretical profit after {NUM_FLIPS} flips: ${theoretical_profit:.2f}")
print(f"Average profit across all simulations: ${avg_final_profit:.2f}")
print(f"Profitable simulations: {profitable_sims}/{NUM_SIMULATIONS} ({profit_percentage:.1f}%)")
print(f"Min profit: ${np.min(final_profits):.2f}")
print(f"Max profit: ${np.max(final_profits):.2f}")
print(f"{'='*60}\n")

fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=(
        'Cumulative Profit Over Time (Multiple Simulations)',
        'Distribution of Final Profits'
    ),
    vertical_spacing=0.12,
    row_heights=[0.6, 0.4]
)

num_to_plot = min(20, NUM_SIMULATIONS)
x_values = np.arange(NUM_FLIPS)

for i in range(num_to_plot):
    fig.add_trace(
        go.Scatter(
            x=x_values,
            y=all_simulations[i],
            mode='lines',
            name=f'Sim {i+1}',
            line=dict(width=1),
            opacity=0.5,
            showlegend=False
        ),
        row=1, col=1
    )

theoretical_line = EV * x_values
fig.add_trace(
    go.Scatter(
        x=x_values,
        y=theoretical_line,
        mode='lines',
        name='Expected Value',
        line=dict(color='red', width=3, dash='dash'),
        showlegend=True
    ),
    row=1, col=1
)

fig.add_hline(y=0, line_dash="dot", line_color="gray", opacity=0.5, row=1, col=1)

fig.add_trace(
    go.Histogram(
        x=final_profits,
        nbinsx=30,
        name='Final Profits',
        marker_color='lightblue',
        showlegend=False
    ),
    row=2, col=1
)
# Took out the vertical avergage final profit as it doesn't provide that much information anyways
# fig.add_vline(
#    x=avg_final_profit,
#    line_dash="dash",
#    line_color="blue",
#    annotation_text=f"Avg: ${avg_final_profit:.0f}",
#    annotation_position="top",
#    row=2, col=1
#)

fig.add_vline(
    x=theoretical_profit,
    line_dash="dash",
    line_color="red",
    annotation_text=f"Expected: ${theoretical_profit:.0f}",
    annotation_position="top",
    row=2, col=1
)

fig.update_xaxes(title_text="Number of Flips", row=1, col=1)
fig.update_yaxes(title_text="Cumulative Profit ($)", row=1, col=1)
fig.update_xaxes(title_text="Final Profit ($)", row=2, col=1)
fig.update_yaxes(title_text="Frequency", row=2, col=1)

fig.update_layout(
    height=900,
    title_text=f"Monte Carlo Simulation: Coin Flip Game<br><sub>Cost: ${COST} | Heads Win: ${PAYOUT_HEADS} | EV: ${EV}/flip | {NUM_SIMULATIONS} simulations × {NUM_FLIPS} flips</sub>",
    showlegend=True,
    hovermode='closest'
)

fig.show()

Running 1000 simulations with 10 flips each...

RESULTS:
Expected Value per flip: $-1.00
Theoretical profit after 10 flips: $-10.00
Average profit across all simulations: $4.90
Profitable simulations: 828/1000 (82.8%)
Min profit: $-10.00
Max profit: $20.00



###  Law of Large Numbers
#### <u>Random Variables</u>

Random variables $X$ are characterized fully by their probability mass or density function, or cumulative distribution function.

It is often confusing to students as $X$ seems like a traditional variable when in reality it is a *distribution* defined by $D$.

Different draws from $X$ will yield different values but drawing *a lot* of values and plotting a histogram will yield the distribution $D$ that governs the randomness of $X$.

*Remark:* Different types of convergence exist in probability theory - convergence in distribution (weakest), convergence in probability (stronger), and almost sure convergence (strongest). Each describes how random variables approach a limit in different ways.


In [3]:
### Adapted from Why Monte Carlo Simulations Work
### https://github.com/romanmichaelpaolucci/Quant-Guild-Library/tree/757ea6e23dd2f007d5799f1682450b5ebb20b066/2025%20Video%20Lectures/33.%20Why%20Monte%20Carlo%20Simulation%20Works


import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

np.random.seed(42)
x = np.linspace(-4, 4, 1000)
theoretical_dist = stats.norm.pdf(x)

fig = go.Figure()

fig.add_trace(
    go.Histogram(
        x=[],
        histnorm='probability density',
        name='Empirical Distribution',
        nbinsx=50,
        opacity=0.7
    )
)

fig.add_trace(
    go.Scatter(
        x=x,
        y=theoretical_dist,
        name='Theoretical Distribution',
        line=dict(color='red')
    )
)

frames = []
sample_sizes = [10, 50, 100, 500, 1000, 5000, 10000]

for n in sample_sizes:
    samples = np.random.normal(0, 1, n)

    frame = go.Frame(
        data=[
            go.Histogram(
                x=samples,
                histnorm='probability density',
                name='Empirical Distribution',
                nbinsx=50,
                opacity=0.7
            ),
            go.Scatter(
                x=x,
                y=theoretical_dist,
                name='Theoretical Distribution',
                line=dict(color='red')
            )
        ],
        name=f'n={n}'
    )
    frames.append(frame)

fig.frames = frames

fig.update_layout(
    title='Empirical Convergence',
    xaxis_title='Value',
    yaxis_title='Density',
    showlegend=True,
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)',
    font=dict(color='white'),
    updatemenus=[{
        'type': 'buttons',
        'showactive': False,
        'buttons': [
            {'label': 'Play',
             'method': 'animate',
             'args': [None, {'frame': {'duration': 1000, 'redraw': True},
                            'fromcurrent': True}]},
            {'label': 'Pause',
             'method': 'animate',
             'args': [[None], {'frame': {'duration': 0, 'redraw': False},
                              'mode': 'immediate',
                              'transition': {'duration': 0}}]}
        ]
    }],
    sliders=[{
        'currentvalue': {'prefix': 'Sample Size: ', 'font': {'color': 'white'}},
        'steps': [
            {'label': str(n),
             'method': 'animate',
             'args': [[f'n={n}'], {'frame': {'duration': 0, 'redraw': True},
                                  'mode': 'immediate',
                                  'transition': {'duration': 0}}]}
            for n in sample_sizes
        ]
    }]
)

fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='rgba(128,128,128,0.2)',
                 zeroline=True, zerolinewidth=1, zerolinecolor='rgba(128,128,128,0.5)')
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='rgba(128,128,128,0.2)',
                 zeroline=True, zerolinewidth=1, zerolinecolor='rgba(128,128,128,0.5)')

fig.show()